### This Jupyter Notebook preprocesses and cleans the input seafood data.
### It goes through the following steps :
#### 0. Checks for input files
#### 1. (Preprocess) Extracts and normalizes relevant HS Codes (03,1604,1605)
#### 2. (Preprocess) Retains only relevant columns
#### 3. (Preprocess) Combines individual month files into one dataset
#### 4. (Preprocess) Standardizes consignee and shipper names
#### 5. (Cleaning) Condenses rows based on reference columns
#### 6. (Cleaning) Clusters rows based on clustering logic

#### 

#### 0. Checks for files in input folder - us_imports_2015

In [2]:
from pathlib import Path
import re

# Point to your input folder
input_folder = Path.cwd().parent / "input" / "us_imports_2015"
print("Using folder:", input_folder)
print("Exists?", input_folder.exists(), "Is dir?", input_folder.is_dir())

# Regex to extract the first number in a filename
def extract_number(path: Path):
    match = re.search(r"\d+", path.stem)  # look at filename without extension
    return int(match.group()) if match else float("inf")

# Collect and sort CSV files by number in filename
csv_paths = sorted(input_folder.rglob("*.csv"), key=extract_number)

print("CSV files found:", len(csv_paths))
for p in csv_paths:
    print("-", p.name)


Using folder: c:\Users\brant\LLM_Research\DataCleanse\input\us_imports_2015
Exists? True Is dir? True
CSV files found: 1
- panjiva_us_imports_02_2015.csv


####

#### 1. (Preprocess) Extracts and normalizes relevant HS Codes (03,1604,1605).
#### Filters rows to keep only those with HS codes starting with "03", "1604", or "1605". Normalizes the HS tokens (fixing leading zero issues, ensuring 6/8/10-digit length)

In [3]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, Tuple, List
import re
import pandas as pd
import csv

# Paths 
folder = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015")
out_root = folder / "hs_code"
out_root.mkdir(parents=True, exist_ok=True)


# Configuration Set Up
CHUNK_ROWS = 150_000  # read in chunks for large files
HS_CANDIDATE_NAMES: List[str] = [
    "HSCode", "HS Code", "HS_Code", "HS", "HSCODE", "HTS", "HTSCode", "HTS Code",
]
DIGIT_RE = re.compile(r"\D+")         # match non-digits
NUM_IN_STEM_RE = re.compile(r"(\d+)") # extract trailing numbers from filenames


# Functions
def normalize_hs_token(token: str) -> Optional[str]:
    """
    Normalize an HS token to a 6/8/10-digit code:
      - Remove non-digit characters.
      - Prepend '0' if starts with '3' but not '03'.
      - Only accept lengths 6,8,10.
    """
    if token is None:
        return None
    digits = DIGIT_RE.sub("", str(token))
    if not digits:
        return None
    if digits.startswith("3") and not digits.startswith("03"):
        digits = "0" + digits
    if len(digits) in (5, 7, 9):
        digits = "0" + digits
    if len(digits) not in (6, 8, 10):
        return None
    return digits

def _normalize_cell_to_tokens(cell: str) -> List[str]:
    """
    Split a cell by commas, normalize each HS code token,
    deduplicate them while preserving order.
    """
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    seen, out = set(), []
    for raw in str(cell).split(","):
        tok = normalize_hs_token(raw.strip())
        if tok and tok not in seen:
            seen.add(tok)
            out.append(tok)
    return out

def any_token_matches_target(cell: str) -> bool:
    """
    Return True if any normalized token starts with the HS prefixes of interest:
    '03', '1604', or '1605'.
    """
    for tok in _normalize_cell_to_tokens(cell):
        if tok.startswith(("03", "1604", "1605")):
            return True
    return False

def score_as_hs_column(series: pd.Series, sample: int = 10_000) -> float:
    s = series.dropna().astype(str).head(sample)
    if s.empty:
        return 0.0
    good = sum(1 for val in s if _normalize_cell_to_tokens(val))
    return good / len(s)

def detect_hs_column(df: pd.DataFrame, min_score: float = 0.30) -> Optional[str]:
    for name in HS_CANDIDATE_NAMES:
        if name in df.columns:
            return name
    scores = {col: score_as_hs_column(df[col]) for col in df.columns}
    if not scores:
        return None
    best_col, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_col if best_score >= min_score else None

def write_filtered_csv(src_path: Path, dest_path: Path, hs_col: str) -> Tuple[int, int]:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    header_written = False
    kept, total = 0, 0
    for chunk in pd.read_csv(src_path, chunksize=CHUNK_ROWS, dtype=str, low_memory=False):
        total += len(chunk)
        mask = chunk[hs_col].apply(any_token_matches_target)
        sub = chunk.loc[mask].copy()
        if not sub.empty:
            sub[hs_col] = sub[hs_col].apply(lambda cell: ",".join(_normalize_cell_to_tokens(cell)))
            sub.to_csv(dest_path, mode="a", header=not header_written, index=False, quoting=csv.QUOTE_NONNUMERIC)
            header_written = True
            kept += len(sub)
    return kept, total

def key_by_number(p: Path):
    """Sort helper: arranges CSV files in numerical order"""
    nums = NUM_IN_STEM_RE.findall(p.stem)
    return (int(nums[-1]) if nums else float("inf"), p.name.lower())

csv_paths = sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower()==".csv"], key=key_by_number)
N = len(csv_paths)

grand_total = 0
grand_kept  = 0
skipped     = 0

for i, p in enumerate(csv_paths, 1):
    out_path = out_root / f"{p.stem}_filtered.csv"

    # Detect HS column using a sample
    try:
        sample_df = pd.read_csv(p, nrows=50_000, dtype=str, low_memory=False)
    except Exception:
        print(f"{i}/{N} {p.name} → kept: 0 | input: 0 | removed: 0 (read error)")
        skipped += 1
        continue

    hs_col = detect_hs_column(sample_df)
    if not hs_col:
        print(f"{i}/{N} {p.name} → kept: 0 | input: 0 | removed: 0 (no HS column)")
        skipped += 1
        continue

    kept, total = write_filtered_csv(p, out_path, hs_col)
    removed = total - kept
    grand_total += total
    grand_kept  += kept

    print(f"{i}/{N} {p.name} → kept: {kept:,} | input: {total:,} | removed: {removed:,}")

overall_removed = grand_total - grand_kept
print(f"\nAll files saved under: {out_root}")
print(f"Totals — kept: {grand_kept:,} | input: {grand_total:,} | removed: {overall_removed:,}")


# Dataframes
if csv_paths:
    first_out = out_root / f"{csv_paths[0].stem}_filtered.csv"
    if first_out.exists():
        print(f"\nPreview of first saved file: {first_out.name}")
        df_preview = pd.read_csv(first_out, dtype=str, nrows=10)
        display(df_preview)
    else:
        print("\nNo filtered file found for preview.")
else:
    print("\nNo input CSVs were processed, nothing to preview.")


1/1 panjiva_us_imports_02_2015.csv → kept: 7,898 | input: 802,028 | removed: 794,130

All files saved under: \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\hs_code
Totals — kept: 7,898 | input: 802,028 | removed: 794,130

Preview of first saved file: panjiva_us_imports_02_2015_filtered.csv


,PanjivaRecordID,BillOfLadingNumber,ArrivalDate,DataLoadDate,DataLaunchDate,ConsigneeName,ConsigneeFullAddress,ConsigneeRoute,ConsigneeCity,ConsigneeStateRegion,...,HasLCL,ContainerNumbers,HSCode,GoodsShipped,VolumeContainerTEU,ContainerMarks,DividedLCL,ContainerTypeOfService,ContainerTypes,DangerousGoods
0,107686654,COSU6107751810,2015-02-04,2015-02-05,2015-02-07,Fulton Seafood Inc.,2818 MCKINNEY ST HOUSTON TEXAS 77003,2818 McKinney Street,Houston,Texas,...,NaN,CBHU2817521,030323,FROZEN GUTTED AND SCALED TILAPIA,2.0,NaN,N,House to House,4532,false
1,107644350,OWLQHK1400485,2015-02-03,2015-02-04,2015-02-06,Quirch Foods,7600 NW 82ND PLACE MIAMI,NaN,NaN,NaN,...,NaN,SEGU9052257,030461,FROZEN TILAPIA FILLETS PO NO. 394814 HS CODE IUS,2.0,030461 CONTAINER IS SET AT -18 DEGREES CELS,N,Pier to Pier,NaN,false
2,107690878,EGLV149403838133,2015-02-04,2015-02-05,2015-02-07,Obaba Seafood Co. Inc.,641 BRENNAN ST SAN JOSE CA 95131 USA,641 Brennan Street,San Jose,California,...,NaN,TCLU1230644,030461,FROZEN TILAPIA FILLET FROZEN TILAPIA FILLET ...,2.0,NaN,N,House to House,45R1,false
3,107794238,EIMSDLCSAV000755,2015-02-09,2015-02-10,2015-02-12,Aquanita Foods Llc,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",2140 South Dixie Highway,Miami,Florida,...,NaN,CCLU8613124,030429,FROZEN ATLANTIC RED FISH FILLET (SEBASTES MEN .,2.0,TELLA),N,Pier to Pier,NaN,false
4,107729790,SEYOPAN1412258,2015-02-03,2015-02-05,2015-02-07,High Liner Foods Inc.,"100 BATTERY POINT ROAD LUNENBURG, NS B0J 2C0",NaN,NaN,NaN,...,NaN,SZLU9087088,030481,FROZEN WILD PACIFIC PINK SALMON FILLETS,2.0,NaN,N,Container Yard,4FR0,false
5,108058430,EZLOSTMI15010006,2015-02-17,2015-02-19,2015-02-21,Panapesca Usa Corp.,"42 WINTER STREET UNIT 7 PEMBROKE, MA 02359",42 Winter Street,Pembroke,Massachusetts,...,NaN,MWMU6354795,030749,FROZEN CLEANED LOLIGO SQUID SCIENTIFIC NAME: ...,2.0,NaN,N,Break Bulk,45R1,false
6,108229310,EZLOJUSO15010007,2015-02-20,2015-02-21,2015-02-23,Jh Seafood Supply,2001 SANTA ANITA AVENUE SUITE 202 SOUTH EL MON...,2001 Santa Anita Avenue,South El Monte,California,...,NaN,MSWU4000509,160420,430 CARTONS OF FROZEN IMITATION CRABMEAT SURIM...,2.0,NaN,N,Container Yard,45R1,false
7,108202750,ZIMUQIN4504723,2015-02-16,2015-02-17,2015-02-19,Gracekennedy Foods Usa Llc,230 MOONACHIE AVE.MOONACHIE NJ 07074 NEW YORK ...,230 Moonachie Avenue,Moonachie,New Jersey,...,NaN,GLDU5272152,160413,GR.CANNED MACKEREL IN TOMATO SAUCE 24X42 5G (...,1.0,NaN,N,House to House,2200,false
8,107984318,APLU051536939,2015-02-13,2015-02-16,2015-02-18,Aquanita Foods Llc,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",2140 South Dixie Highway,Miami,Florida,...,NaN,APRU5780014,030461,FROZEN TILAPIA FILLET -OREOCHROMIS NILOTICUS- ...,2.0,NaN,N,Container Yard,45R0,false
9,107869566,OOLU4018241100,2015-02-08,2015-02-09,2015-02-11,Bumble Bee Foods Llc,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,9655 Granite Ridge Drive,San Diego,California,...,NaN,OOLU1414487,160590,BUMBLE BEE BOILED OYSTERS FACILITY REGISTRATI ON,1.0,NaN,N,House to House,2200,false


####

#### 2. (Preprocess) Retains only relevant columns
#### Retain only relevant reference columns (defined in BUSINESS_KEEP_COLS) and output the files 

In [5]:
from pathlib import Path
from typing import Optional, Tuple
import re
import pandas as pd

# Paths
in_folder  = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015/hs_code")
out_folder = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015/columns")
out_folder.mkdir(parents=True, exist_ok=True)

CHUNK_ROWS = 150_000  # process in chunks to handle large files efficiently

# Columns to keep if present
BUSINESS_KEEP_COLS = [
    # Consignee
    "ConsigneeName", "ConsigneeFullAddress", "ConsigneeLocalDUNS",
    "ConsigneePanjivaID", "ConsigneeOriginalFormat",

    # Shipper
    "ShipperName", "ShipperFullAddress", "ShipperLocalDUNS",
    "ShipperPanjivaID", "ShipperOriginalFormat",

    # Weight / quantity variants
    "GrossWeightKg", "NetWeightKg", "WeightKg", "Weightkg",
    "Quantity (kg)", "Quantity_kg", "QuantityKg",

    # Value (USD) variants
    "ValueUSD", "Value_USD", "FOBUSD", "ValueOfGoodsFOBUSD", "CIFUSD",
    "ExportValue", "ValueOfGoodsUSD", "InvoiceValueUSD", "Value",
]

# Candidate HS column names
HS_CANDIDATE_NAMES = [
    "HSCode", "HS Code", "HS_Code", "HS", "HSCODE", "HTS", "HTSCode", "HTS Code",
]

# Functions
def _score_as_hs_column(series: pd.Series, sample: int = 10_000) -> float:
    s = series.dropna().astype(str).head(sample)
    if s.empty:
        return 0.0
    keep = sum(1 for val in s if re.search(r"\d", val))
    return keep / len(s)

def detect_hs_column(df: pd.DataFrame, min_score: float = 0.30) -> Optional[str]:
    for name in HS_CANDIDATE_NAMES:
        if name in df.columns:
            return name
    scores = {col: _score_as_hs_column(df[col]) for col in df.columns}
    if not scores:
        return None
    best_col, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_col if best_score >= min_score else None

# Ensure CSVs are processed in natural numeric order (01,02,...12)
num_re = re.compile(r"(\d+)")
def sort_key(p: Path):
    nums = num_re.findall(p.stem)
    return (int(nums[-1]) if nums else float("inf"), p.name.lower())

csv_paths = sorted(
    [p for p in in_folder.iterdir() if p.is_file() and p.suffix.lower() == ".csv"],
    key=sort_key
)

def write_slimmed_csv(src: Path, dest: Path) -> Tuple[int, int, int]:
    # Read only the header first
    head = pd.read_csv(src, nrows=0, dtype=str, low_memory=False)
    hs_col = detect_hs_column(head)

    # Fallback: sample rows if not found
    if not hs_col:
        sample = pd.read_csv(src, nrows=50_000, dtype=str, low_memory=False)
        hs_col = detect_hs_column(sample)
    if not hs_col:
        return (0, head.shape[1], 0)

    # Select keep columns
    keep_found = [c for c in BUSINESS_KEEP_COLS if c in head.columns]
    usecols = [hs_col] + [c for c in keep_found if c != hs_col]

    # Stream in chunks and write
    header_written = False
    rows_written = 0
    for chunk in pd.read_csv(src, usecols=usecols, chunksize=CHUNK_ROWS, dtype=str, low_memory=False):
        chunk.to_csv(dest, mode="a", header=not header_written, index=False)
        header_written = True
        rows_written += len(chunk)

    return (rows_written, head.shape[1], len(usecols))

grand_rows = 0
first_output_path = None
in_cols_final = None
out_cols_final = None

for i, p in enumerate(csv_paths, 1):
    out_path = out_folder / f"{p.stem}_cols.csv"
    rows, in_cols, out_cols = write_slimmed_csv(p, out_path)
    grand_rows += rows
    if first_output_path is None and rows > 0:
        first_output_path = out_path
        in_cols_final = in_cols
        out_cols_final = out_cols

print("\nFiles saved under:", out_folder)
print(f"Total rows across all outputs: {grand_rows:,}")
if in_cols_final is not None and out_cols_final is not None:
    print(f"Number of columns decreased from {in_cols_final} → {out_cols_final}")

if first_output_path and first_output_path.exists():
    print("\nPreview of first edited file:", first_output_path.name)
    df_preview = pd.read_csv(first_output_path, nrows=10, dtype=str, low_memory=False)
    display(df_preview)
else:
    print("\n(No non-empty outputs to preview.)")
    


Files saved under: \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\columns
Total rows across all outputs: 7,898
Number of columns decreased from 77 → 13

Preview of first edited file: panjiva_us_imports_02_2015_filtered_cols.csv


,ConsigneeName,ConsigneeFullAddress,ConsigneeLocalDUNS,ConsigneePanjivaID,ConsigneeOriginalFormat,ShipperName,ShipperFullAddress,ShipperLocalDUNS,ShipperPanjivaID,ShipperOriginalFormat,WeightKg,ValueOfGoodsUSD,HSCode
0,Fulton Seafood Inc.,2818 MCKINNEY ST HOUSTON TEXAS 77003,137544938,1982366,"FULTON SEAFOOD, INC. 2818 MCKINNEY STREET HOUS...",Xiamen Huison Foods Co.,NO.1339 TONGJI RD TONGJI INDUSTRIAL ZONE XIAME...,NaN,45342071,XIAMEN HUISON FOODS COMPANY LIMITED 1339 TONGJ...,21275.0,44500.0,030323
1,Quirch Foods,7600 NW 82ND PLACE MIAMI,045467826,27824514,QUIRCH FOODS 7600 NW 82ND PLACE MIAMI FL 33016...,Guangxi Nanning Baiyang Food Co.,CO. LTD. NO. 16 CHUANGXIN XI RD NEW AND HIGH-T...,NaN,5096630,GUANGXI NANNING BAIYANG FOOD CO LTD NO 16 CHUA...,23625.0,107000.0,030461
2,Obaba Seafood Co. Inc.,641 BRENNAN ST SAN JOSE CA 95131 USA,060625169,5774939,"OBABA SEAFOOD COMPANY,INC. 641 BRENNAN STREET ...",Xinxing Aquatic Products Processing,FACTORY MAOAN INDUSTRIAL BASE JI DONG YI XIAOL...,NaN,29060263,XINXING AQUATIC PRODUCTS PROCESSING FACTORY MA...,20476.0,92500.0,030461
3,Aquanita Foods Llc,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",045467826,4686456,AQUANITA FOODS LLC 2140 SOUTH DIXIE HWY SUITE ...,Dahuachem International Economic,AND TRADE CORP NO 3 GANXIN ST GANGJINGZI DISTR...,654505429,44214093,DAHUACHEM INTL ECONOMIC&TRADE CORP NO.3 GANXIN...,24550.0,NaN,030429
4,High Liner Foods Inc.,"100 BATTERY POINT ROAD LUNENBURG, NS B0J 2C0",200271849,27846754,"HIGH LINER FOODS INC. PO BOX 910, 100 BARTTERY...",Dalian Oceanstone (Zf Group) Foods,CO. LTD. HOUHAI VILLAGE QIDINGSHAN XIANG JINZH...,NaN,44704138,"DALIAN OCEANSTONE (ZF GROUP) FOODS NO.1, HOUHA...",23290.0,146000.0,030481
5,Panapesca Usa Corp.,"42 WINTER STREET UNIT 7 PEMBROKE, MA 02359",804891273,44225830,"PANAPESCA USA LLC 42 WINTER STREET UNIT 7,PEMB...",Zhangzhou Fuhai Food,"LINTOU CUN ZHANAN COUNTY ZHANGZHOU, FUJIAN",NaN,33649517,"ZHANGZHOU FUHAI FOOD CO.,LTD LINTOU CUN,SIDU T...",22550.0,78400.0,030749
6,Jh Seafood Supply,2001 SANTA ANITA AVENUE SUITE 202 SOUTH EL MON...,079614278,36260029,JH SEAFOOD SUPPLY INC. 2001 SANTA ANITA AVE. S...,"Shantou City Qiaofeng Group Co., Ltd.","Harbour Dock, Gucheng Jindu Chaon Zone, Shanto...",527193336,1769168,"SHANTOU CITY QIAOFENG GROUP CO.,LTD HARBOUR DO...",18632.0,43600.0,160420
7,Gracekennedy Foods Usa Llc,230 MOONACHIE AVE.MOONACHIE NJ 07074 NEW YORK ...,079492994,45857811,GRACEKENNEDY FOODS (USA) LLC A DIVISION OF GRA...,"Rongcheng Huiying Foods Co., Ltd.",NO.3 SHIDAO INDUSTRIAL ZONE RONGCHENG 264300 CN,NaN,30433241,"RONGCHENG HUIYING FOODS CO.,LTD. NO.3 SHIDAO I...",8981.13,19700.0,160413
8,Aquanita Foods Llc,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",045467826,4686456,AQUANITA FOODS LLC 2140 SOUTH DIXIE HWY SUITE ...,Huazhou Xinhai Aquatic Products,CO. LTD. PUSHAN VILLAGE NANSHENG ST HUAZHOU CI...,NaN,44439162,HUAZHOU XINHAI AQUATIC PRODUCTS CO. LTD PUSHAN...,22998.0,104000.0,030461
9,Bumble Bee Foods Llc,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,135952609,27818233,"BUMBLE BEE FOODS, LLC 280 10TH AVE SAN DIEGO C...",Qingdao Ocean Garden Imp. & Exp.,CO. LTD. HUAYU MANSION NO.48 SHANDONG RD QINGD...,NaN,28855377,QINGDAO OCEAN GARDEN IMPORT AND EXP NO.52 SHAN...,17094.0,NaN,160590


####

#### 3. (Preprocess) Combines individual month files into one dataset
#### After the first two preprocessing steps, the individual month files are combined into one final output for easier standardization

In [6]:
import pandas as pd
from pathlib import Path
import re

# Paths
base_folder  = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015")
input_folder = base_folder / "columns"
output_file  = base_folder / "us_import_2015_combined.csv"

csv_files = list(input_folder.glob("*.csv"))

# Functions
def numerical_sort(path: Path):
    """Extract numbers from filename for natural numeric ordering."""
    nums = re.findall(r"\d+", path.stem)
    return [int(n) for n in nums] if nums else [float("inf")]

csv_files = sorted(csv_files, key=numerical_sort)
print(f"Found {len(csv_files)} CSV files in {input_folder}")

# Read and combine 
df_list = []
for f in csv_files:
    # Read each CSV as string-typed DataFrame (safe for IDs, codes, etc.)
    df = pd.read_csv(f, dtype=str, low_memory=False)
    df_list.append(df)

# Concatenate all DataFrames into one
combined_df = pd.concat(df_list, ignore_index=True)

output_file.parent.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(output_file, index=False)

print(f"\nFinal combined file saved to: {output_file}")
print(f"Total rows: {combined_df.shape[0]:,}")
print(f"Total cols: {combined_df.shape[1]:,}")

display(combined_df.head(10))


Found 1 CSV files in \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\columns

Final combined file saved to: \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\us_import_2015_combined.csv
Total rows: 15,797
Total cols: 13


,ConsigneeName,ConsigneeFullAddress,ConsigneeLocalDUNS,ConsigneePanjivaID,ConsigneeOriginalFormat,ShipperName,ShipperFullAddress,ShipperLocalDUNS,ShipperPanjivaID,ShipperOriginalFormat,WeightKg,ValueOfGoodsUSD,HSCode
0,Fulton Seafood Inc.,2818 MCKINNEY ST HOUSTON TEXAS 77003,137544938,1982366,"FULTON SEAFOOD, INC. 2818 MCKINNEY STREET HOUS...",Xiamen Huison Foods Co.,NO.1339 TONGJI RD TONGJI INDUSTRIAL ZONE XIAME...,NaN,45342071,XIAMEN HUISON FOODS COMPANY LIMITED 1339 TONGJ...,21275.0,44500.0,030323
1,Quirch Foods,7600 NW 82ND PLACE MIAMI,045467826,27824514,QUIRCH FOODS 7600 NW 82ND PLACE MIAMI FL 33016...,Guangxi Nanning Baiyang Food Co.,CO. LTD. NO. 16 CHUANGXIN XI RD NEW AND HIGH-T...,NaN,5096630,GUANGXI NANNING BAIYANG FOOD CO LTD NO 16 CHUA...,23625.0,107000.0,030461
2,Obaba Seafood Co. Inc.,641 BRENNAN ST SAN JOSE CA 95131 USA,060625169,5774939,"OBABA SEAFOOD COMPANY,INC. 641 BRENNAN STREET ...",Xinxing Aquatic Products Processing,FACTORY MAOAN INDUSTRIAL BASE JI DONG YI XIAOL...,NaN,29060263,XINXING AQUATIC PRODUCTS PROCESSING FACTORY MA...,20476.0,92500.0,030461
3,Aquanita Foods Llc,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",045467826,4686456,AQUANITA FOODS LLC 2140 SOUTH DIXIE HWY SUITE ...,Dahuachem International Economic,AND TRADE CORP NO 3 GANXIN ST GANGJINGZI DISTR...,654505429,44214093,DAHUACHEM INTL ECONOMIC&TRADE CORP NO.3 GANXIN...,24550.0,NaN,030429
4,High Liner Foods Inc.,"100 BATTERY POINT ROAD LUNENBURG, NS B0J 2C0",200271849,27846754,"HIGH LINER FOODS INC. PO BOX 910, 100 BARTTERY...",Dalian Oceanstone (Zf Group) Foods,CO. LTD. HOUHAI VILLAGE QIDINGSHAN XIANG JINZH...,NaN,44704138,"DALIAN OCEANSTONE (ZF GROUP) FOODS NO.1, HOUHA...",23290.0,146000.0,030481
5,Panapesca Usa Corp.,"42 WINTER STREET UNIT 7 PEMBROKE, MA 02359",804891273,44225830,"PANAPESCA USA LLC 42 WINTER STREET UNIT 7,PEMB...",Zhangzhou Fuhai Food,"LINTOU CUN ZHANAN COUNTY ZHANGZHOU, FUJIAN",NaN,33649517,"ZHANGZHOU FUHAI FOOD CO.,LTD LINTOU CUN,SIDU T...",22550.0,78400.0,030749
6,Jh Seafood Supply,2001 SANTA ANITA AVENUE SUITE 202 SOUTH EL MON...,079614278,36260029,JH SEAFOOD SUPPLY INC. 2001 SANTA ANITA AVE. S...,"Shantou City Qiaofeng Group Co., Ltd.","Harbour Dock, Gucheng Jindu Chaon Zone, Shanto...",527193336,1769168,"SHANTOU CITY QIAOFENG GROUP CO.,LTD HARBOUR DO...",18632.0,43600.0,160420
7,Gracekennedy Foods Usa Llc,230 MOONACHIE AVE.MOONACHIE NJ 07074 NEW YORK ...,079492994,45857811,GRACEKENNEDY FOODS (USA) LLC A DIVISION OF GRA...,"Rongcheng Huiying Foods Co., Ltd.",NO.3 SHIDAO INDUSTRIAL ZONE RONGCHENG 264300 CN,NaN,30433241,"RONGCHENG HUIYING FOODS CO.,LTD. NO.3 SHIDAO I...",8981.13,19700.0,160413
8,Aquanita Foods Llc,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",045467826,4686456,AQUANITA FOODS LLC 2140 SOUTH DIXIE HWY SUITE ...,Huazhou Xinhai Aquatic Products,CO. LTD. PUSHAN VILLAGE NANSHENG ST HUAZHOU CI...,NaN,44439162,HUAZHOU XINHAI AQUATIC PRODUCTS CO. LTD PUSHAN...,22998.0,104000.0,030461
9,Bumble Bee Foods Llc,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,135952609,27818233,"BUMBLE BEE FOODS, LLC 280 10TH AVE SAN DIEGO C...",Qingdao Ocean Garden Imp. & Exp.,CO. LTD. HUAYU MANSION NO.48 SHANDONG RD QINGD...,NaN,28855377,QINGDAO OCEAN GARDEN IMPORT AND EXP NO.52 SHAN...,17094.0,NaN,160590


####

#### 4. (Preprocess) Standardizes consignee and shipper names
#### Applies normalization rules to company names, removes unwanted punctuation and unifies spacing

In [4]:
import re, unicodedata
import pandas as pd
from pathlib import Path

# Paths
base = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015")
src_file = base / "us_import_2015_combined.csv"
std_dir  = base / "std"
std_dir.mkdir(parents=True, exist_ok=True)

out_std_file   = std_dir / "us_import_2015_combined_std.csv"
out_audit_file = std_dir / "name_standardization_audit.csv"

assert src_file.exists(), f"Input not found: {src_file}"

# Regex replacements 
# Order matters — more specific patterns must appear first
REPLACEMENTS = [
    (re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\s*de\s*c\.?\s*v\.?\b", re.I), "S. de R.L. de C.V."),
    # Mexican forms
    (re.compile(r"\bs\.?\s*a\.?\s*de\s*c\.?\s*v\.?\b", re.I), "S.A. de C.V."),
    (re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\b", re.I),       "S. de R.L."),
    # English composites
    (re.compile(r"\bpty\s+limited\b", re.I),             "Pty Ltd"),
    (re.compile(r"\bco\.\s*,?\s*ltd\b", re.I),           "Company Limited"),
    (re.compile(r"\bco\s*,?\s*ltd\b", re.I),             "Company Limited"),
    (re.compile(r"\bcompany\s+limited\b", re.I),         "Company Limited"),
    # Single-word variants
    (re.compile(r"\bincorporated\b|\binc\b\.?", re.I),   "Incorporated"),
    (re.compile(r"\bcorporation\b|\bcorp\b\.?", re.I),   "Corporation"),
    (re.compile(r"\bcompany\b|\bco\b\.?", re.I),         "Company"),
    (re.compile(r"\blimited\b|\bltd\b\.?|\bltda\b\.?", re.I), "Limited"),
    # Acronyms (forced upper)
    (re.compile(r"\bplc\b", re.I), "PLC"),
    (re.compile(r"\bllc\b", re.I), "LLC"),
    (re.compile(r"\bllp\b", re.I), "LLP"),
    (re.compile(r"\bgmbh\b", re.I), "GMBH"),
    (re.compile(r"\bag\b", re.I), "AG"),
    (re.compile(r"\bbv\b", re.I), "BV"),
    (re.compile(r"\bnv\b", re.I), "NV"),
    (re.compile(r"\bsa\b", re.I), "SA"),
    (re.compile(r"\bsrl\b", re.I), "SRL"),
    (re.compile(r"\bspa\b", re.I), "SPA"),
    (re.compile(r"\bab\b", re.I), "AB"),  # keep AB uppercase
    # Collapse separated Pty + Ltd
    (re.compile(r"\bpty\b\s+\bltd\b", re.I), "Pty Ltd"),
]

# Functions
PUNCT_TO_SPACE = re.compile(r"[\,\.;:\|\(\)\[\]\{\}/\\\+\=\*\!\?\#\^\"“”‘’`~_–—\-]+")
WS_RE = re.compile(r"\s+")

def _nfkd_lower(s: str) -> str:
    """Normalize to ASCII + lowercase for uniform matching."""
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii").lower()

def _normalize_punct_ws(s: str) -> str:
    """Replace punctuation with spaces, collapse whitespace, keep '&' as 'and'."""
    s = s.replace("&", " and ")
    s = PUNCT_TO_SPACE.sub(" ", s)
    s = WS_RE.sub(" ", s).strip()
    return s

def standardize_name(raw) -> str:
    """Apply regex replacements and fix casing of tokens."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return ""
    # Normalize and clean punctuation/spacing
    s = _nfkd_lower(str(raw))
    s = _normalize_punct_ws(s)
    # Apply regex replacements
    for pat, repl in REPLACEMENTS:
        s = pat.sub(repl, s)
    # Token-level casing rules
    tokens = s.split()
    ALWAYS_UPPER = {"LLC","LLP","PLC","GMBH","AG","BV","NV","SA","SRL","SPA","AB"}
    ACRONYM_WITH_DOTS = {"S.A.", "R.L.", "C.V."}
    KEEP_AS_IS = {"Pty", "Ltd"}
    def nice_case(tok: str) -> str:
        if tok in ACRONYM_WITH_DOTS: return tok
        if tok.upper() in ALWAYS_UPPER: return tok.upper()
        if tok.lower() == "de": return "de"   # Spanish preposition
        if tok in KEEP_AS_IS: return tok
        return tok.title()
    return " ".join(nice_case(t) for t in tokens).strip()

df = pd.read_csv(src_file, dtype=str, low_memory=False)
fields = [c for c in ("ConsigneeName", "ShipperName") if c in df.columns]

# Save originals for comparison
originals = {c: df[c].copy() for c in fields}

# Apply standardization
for c in fields:
    df[c] = df[c].apply(standardize_name)

# Build audit dataframe (only rows that changed)
def build_diffs(col: str):
    before = originals[col].fillna("").astype(str).str.strip()
    after  = df[col].fillna("").astype(str).str.strip()
    mask = before.ne(after)
    return pd.DataFrame({"Field": col, "Before": before[mask], "After": after[mask]})

audit_df = pd.concat([build_diffs(c) for c in fields], ignore_index=True) if fields else pd.DataFrame(columns=["Field","Before","After"])

# Save results 
df.to_csv(out_std_file, index=False)
audit_df.to_csv(out_audit_file, index=False)

# Stats 
print(f"Saved standardized file → {out_std_file}")
print(f"Saved audit file        → {out_audit_file}")
print(f"Rows: {len(df):,} | Cols: {df.shape[1]}")
print(f"Changes recorded: {len(audit_df):,}")

for c in fields:
    old_unique = originals[c].nunique(dropna=True)
    new_unique = df[c].nunique(dropna=True)
    changed_rows = (originals[c] != df[c]).sum()
    print(f"\n{c}: {old_unique:,} unique old names → {new_unique:,} unique new names | Rows changed: {changed_rows:,}")

print("\nPreview audit:")
display(audit_df.head(10))

print("\nPreview standardized dataset:")
display(df.head(10))


Saved standardized file → \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\std\us_import_2015_combined_std.csv
Saved audit file        → \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\std\name_standardization_audit.csv
Rows: 15,797 | Cols: 13
Changes recorded: 18,936

ConsigneeName: 1,465 unique old names → 1,461 unique new names | Rows changed: 10,403

ShipperName: 1,565 unique old names → 1,556 unique new names | Rows changed: 10,457

Preview audit:


,Field,Before,After
0,ConsigneeName,Fulton Seafood Inc.,Fulton Seafood Incorporated
1,ConsigneeName,Obaba Seafood Co. Inc.,Obaba Seafood Company Incorporated
2,ConsigneeName,Aquanita Foods Llc,Aquanita Foods LLC
3,ConsigneeName,High Liner Foods Inc.,High Liner Foods Incorporated
4,ConsigneeName,Panapesca Usa Corp.,Panapesca Usa Corporation
5,ConsigneeName,Gracekennedy Foods Usa Llc,Gracekennedy Foods Usa LLC
6,ConsigneeName,Aquanita Foods Llc,Aquanita Foods LLC
7,ConsigneeName,Bumble Bee Foods Llc,Bumble Bee Foods LLC
8,ConsigneeName,The Great Fish Co.,The Great Fish Company
9,ConsigneeName,Atlapac Corp,Atlapac Corporation



Preview standardized dataset:


,ConsigneeName,ConsigneeFullAddress,ConsigneeLocalDUNS,ConsigneePanjivaID,ConsigneeOriginalFormat,ShipperName,ShipperFullAddress,ShipperLocalDUNS,ShipperPanjivaID,ShipperOriginalFormat,WeightKg,ValueOfGoodsUSD,HSCode
0,Fulton Seafood Incorporated,2818 MCKINNEY ST HOUSTON TEXAS 77003,137544938,1982366,"FULTON SEAFOOD, INC. 2818 MCKINNEY STREET HOUS...",Xiamen Huison Foods Company,NO.1339 TONGJI RD TONGJI INDUSTRIAL ZONE XIAME...,NaN,45342071,XIAMEN HUISON FOODS COMPANY LIMITED 1339 TONGJ...,21275.0,44500.0,030323
1,Quirch Foods,7600 NW 82ND PLACE MIAMI,045467826,27824514,QUIRCH FOODS 7600 NW 82ND PLACE MIAMI FL 33016...,Guangxi Nanning Baiyang Food Company,CO. LTD. NO. 16 CHUANGXIN XI RD NEW AND HIGH-T...,NaN,5096630,GUANGXI NANNING BAIYANG FOOD CO LTD NO 16 CHUA...,23625.0,107000.0,030461
2,Obaba Seafood Company Incorporated,641 BRENNAN ST SAN JOSE CA 95131 USA,060625169,5774939,"OBABA SEAFOOD COMPANY,INC. 641 BRENNAN STREET ...",Xinxing Aquatic Products Processing,FACTORY MAOAN INDUSTRIAL BASE JI DONG YI XIAOL...,NaN,29060263,XINXING AQUATIC PRODUCTS PROCESSING FACTORY MA...,20476.0,92500.0,030461
3,Aquanita Foods LLC,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",045467826,4686456,AQUANITA FOODS LLC 2140 SOUTH DIXIE HWY SUITE ...,Dahuachem International Economic,AND TRADE CORP NO 3 GANXIN ST GANGJINGZI DISTR...,654505429,44214093,DAHUACHEM INTL ECONOMIC&TRADE CORP NO.3 GANXIN...,24550.0,NaN,030429
4,High Liner Foods Incorporated,"100 BATTERY POINT ROAD LUNENBURG, NS B0J 2C0",200271849,27846754,"HIGH LINER FOODS INC. PO BOX 910, 100 BARTTERY...",Dalian Oceanstone Zf Group Foods,CO. LTD. HOUHAI VILLAGE QIDINGSHAN XIANG JINZH...,NaN,44704138,"DALIAN OCEANSTONE (ZF GROUP) FOODS NO.1, HOUHA...",23290.0,146000.0,030481
5,Panapesca Usa Corporation,"42 WINTER STREET UNIT 7 PEMBROKE, MA 02359",804891273,44225830,"PANAPESCA USA LLC 42 WINTER STREET UNIT 7,PEMB...",Zhangzhou Fuhai Food,"LINTOU CUN ZHANAN COUNTY ZHANGZHOU, FUJIAN",NaN,33649517,"ZHANGZHOU FUHAI FOOD CO.,LTD LINTOU CUN,SIDU T...",22550.0,78400.0,030749
6,Jh Seafood Supply,2001 SANTA ANITA AVENUE SUITE 202 SOUTH EL MON...,079614278,36260029,JH SEAFOOD SUPPLY INC. 2001 SANTA ANITA AVE. S...,Shantou City Qiaofeng Group Company Limited,"Harbour Dock, Gucheng Jindu Chaon Zone, Shanto...",527193336,1769168,"SHANTOU CITY QIAOFENG GROUP CO.,LTD HARBOUR DO...",18632.0,43600.0,160420
7,Gracekennedy Foods Usa LLC,230 MOONACHIE AVE.MOONACHIE NJ 07074 NEW YORK ...,079492994,45857811,GRACEKENNEDY FOODS (USA) LLC A DIVISION OF GRA...,Rongcheng Huiying Foods Company Limited,NO.3 SHIDAO INDUSTRIAL ZONE RONGCHENG 264300 CN,NaN,30433241,"RONGCHENG HUIYING FOODS CO.,LTD. NO.3 SHIDAO I...",8981.13,19700.0,160413
8,Aquanita Foods LLC,"2140 SOUTH DIXIE HIGHWAY SUITE 309 MIAMI, FL 3...",045467826,4686456,AQUANITA FOODS LLC 2140 SOUTH DIXIE HWY SUITE ...,Huazhou Xinhai Aquatic Products,CO. LTD. PUSHAN VILLAGE NANSHENG ST HUAZHOU CI...,NaN,44439162,HUAZHOU XINHAI AQUATIC PRODUCTS CO. LTD PUSHAN...,22998.0,104000.0,030461
9,Bumble Bee Foods LLC,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,135952609,27818233,"BUMBLE BEE FOODS, LLC 280 10TH AVE SAN DIEGO C...",Qingdao Ocean Garden Imp And Exp,CO. LTD. HUAYU MANSION NO.48 SHANDONG RD QINGD...,NaN,28855377,QINGDAO OCEAN GARDEN IMPORT AND EXP NO.52 SHAN...,17094.0,NaN,160590


####

#### 5. (Cleaning) Condenses rows based on reference columns
#### Use reference IDs (ConsigneeLocalDUNS, ConsigneePanjivaID, ShipperPanjivaID)to identify rows that belong to the same underlying entity, even if names differ.

In [5]:
import pandas as pd
from pathlib import Path

#  Paths 
base_src = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015/std")
src_file = base_src / "us_import_2015_combined_std.csv"

base_out = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015/condense")
base_out.mkdir(parents=True, exist_ok=True)

out_df            = base_out / "us_import_2015_combined_std_combined.csv"
out_consig_report = base_out / "consignee_name_groups.csv"
out_ship_report   = base_out / "shipper_name_groups.csv"

df = pd.read_csv(src_file, dtype=str, low_memory=False)
n0 = len(df)

# Ensure reference and name columns exist and are clean strings
for c in ["ConsigneeLocalDUNS","ConsigneePanjivaID","ShipperPanjivaID",
          "ConsigneeName","ShipperName"]:
    if c not in df.columns:
        df[c] = ""
    else:
        df[c] = df[c].fillna("").astype(str).str.strip()

#  DSU (Union-Find) 
class DSU:
    """Disjoint Set Union for grouping by shared IDs."""
    def __init__(self, n): 
        self.p = list(range(n))
        self.r = [0]*n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1

def canonical_by_frequency(names):
    """Pick most frequent name; tie-break lexicographically."""
    s = pd.Series(names)
    vc = s.value_counts(dropna=False)
    top = vc.max()
    return sorted(vc[vc == top].index.tolist())[0]

#  Consignee grouping (by DUNS and/or PanjivaID) 
n = len(df)
by_duns, by_pid = {}, {}
for i,(duns,pid) in enumerate(zip(df["ConsigneeLocalDUNS"], df["ConsigneePanjivaID"])):
    if duns: by_duns.setdefault(duns,[]).append(i)
    if pid:  by_pid.setdefault(pid,[]).append(i)

cons_dsu = DSU(n)
for idxs in list(by_duns.values()) + list(by_pid.values()):
    for j in idxs[1:]:
        cons_dsu.union(idxs[0], j)

cons_comps = {}
for i in range(n):
    cons_comps.setdefault(cons_dsu.find(i), []).append(i)

cons_records = []
for idxs in cons_comps.values():
    names = [df.loc[i,"ConsigneeName"] for i in idxs]
    canonical = canonical_by_frequency(names)
    for i in idxs: 
        df.at[i,"ConsigneeName"] = canonical
    distinct_names = sorted(pd.unique(pd.Series(names)))
    cons_records.append({
        "distinct_names": len(distinct_names),
        "name_list": "; ".join(distinct_names),   # CSV-friendly
        "canonical_name": canonical,
        "consignee_duns": "; ".join(sorted({df.loc[i,"ConsigneeLocalDUNS"] for i in idxs if df.loc[i,"ConsigneeLocalDUNS"]})),
        "consignee_panjiva_ids": "; ".join(sorted({df.loc[i,"ConsigneePanjivaID"] for i in idxs if df.loc[i,"ConsigneePanjivaID"]})),
    })
cons_report = pd.DataFrame(cons_records)

#  Shipper grouping (by ShipperPanjivaID) 
by_ship = {}
for i, pid in enumerate(df["ShipperPanjivaID"]):
    if pid: by_ship.setdefault(pid,[]).append(i)

ship_dsu = DSU(n)
for idxs in by_ship.values():
    for j in idxs[1:]:
        ship_dsu.union(idxs[0], j)

ship_comps = {}
for i in range(n):
    ship_comps.setdefault(ship_dsu.find(i), []).append(i)

ship_records = []
for idxs in ship_comps.values():
    names = [df.loc[i,"ShipperName"] for i in idxs]
    canonical = canonical_by_frequency(names)
    for i in idxs: 
        df.at[i,"ShipperName"] = canonical
    distinct_names = sorted(pd.unique(pd.Series(names)))
    ship_records.append({
        "distinct_names": len(distinct_names),
        "name_list": "; ".join(distinct_names),   # CSV-friendly
        "canonical_name": canonical,
        "shipper_panjiva_ids": "; ".join(sorted({df.loc[i,"ShipperPanjivaID"] for i in idxs if df.loc[i,"ShipperPanjivaID"]})),
    })
ship_report = pd.DataFrame(ship_records)

df_out = df.drop_duplicates().reset_index(drop=True)

df_out.to_csv(out_df, index=False)
cons_report.to_csv(out_consig_report, index=False)
ship_report.to_csv(out_ship_report, index=False)

print(f"Saved dataset   : {out_df}")
print(f"Saved consignee : {out_consig_report}")
print(f"Saved shipper   : {out_ship_report}")
print(f"Input rows: {n0:,} -> Final rows: {len(df_out):,}")
print(f"Consignee groups: {len(cons_report):,}")
print(f"Shipper groups  : {len(ship_report):,}")

def _display_table(df_in: pd.DataFrame, cols, title: str, n: int = 10):
    """Slice BEFORE styling; hide index across pandas versions; wrap long text."""
    print(f"\n{title}")
    subset = df_in.loc[:, [c for c in cols if c in df_in.columns]].head(n)
    styler = subset.style.set_properties(**{"white-space": "pre-wrap"})
    # pandas >= 2.1: Styler.hide(axis="index"), older: hide_index()
    if hasattr(styler, "hide"):
        styler = styler.hide(axis="index")
    elif hasattr(styler, "hide_index"):
        styler = styler.hide_index()
    display(styler)

# Consignee: show reconciled groups, sorted descending
_cons_prev = cons_report.copy()
_cons_prev["distinct_names"] = pd.to_numeric(_cons_prev["distinct_names"], errors="coerce").fillna(0).astype(int)
if (_cons_prev["distinct_names"] > 1).any():
    _cons_prev = _cons_prev[_cons_prev["distinct_names"] > 1]
_cons_prev = _cons_prev.sort_values(["distinct_names","canonical_name"], ascending=[False, True]).copy()
_cons_prev["name_list"] = _cons_prev["name_list"].astype(str).str.replace("; ", "\n")  
_display_table(
    _cons_prev,
    ["distinct_names","canonical_name","name_list","consignee_duns","consignee_panjiva_ids"],
    title="Preview consignee groups",
    n=10
)

# Shipper: show first 10 rows 
_ship_prev = ship_report.copy()
_ship_prev["name_list"] = _ship_prev["name_list"].astype(str).str.replace("; ", "\n") 
_display_table(
    _ship_prev,
    ["distinct_names","canonical_name","name_list","shipper_panjiva_ids"],
    title="Preview shipper groups",
    n=10
)


Saved dataset   : \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\condense\us_import_2015_combined_std_combined.csv
Saved consignee : \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\condense\consignee_name_groups.csv
Saved shipper   : \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\condense\shipper_name_groups.csv
Input rows: 15,797 -> Final rows: 6,297
Consignee groups: 2,180
Shipper groups  : 2,837

Preview consignee groups


distinct_names,canonical_name,name_list,consignee_duns,consignee_panjiva_ids
3,Chicken Of The Sea Frozen Foods,Chicken Of The Sea Frozen Foods Thai Union Frozen Product Public Tri Union Frozen Products Incorporated,001706167,29039261; 44470413; 45984624
2,Aqua Star Usa Corporation,Aqua Star Aqua Star Usa Corporation,046704685,33483713; 46006229
2,Arko Food International Incorporated,Arko Food International Incorporated Asian Commodities,038672291,1861645; 45207474
2,Great American Seafood,Great American Seafood Southwind Foods LLC Dba,104455618,27840486; 27973838
2,Gurrentz International,Galaxy International Gurrentz International,004504122,32411541; 40037567
2,Hanover Foods Sunwise,Hanover Foods Hanover Foods Sunwise,003004439,33673720; 42244844
2,King And Prince Seafood Corporation,King And Price King And Prince Seafood Corporation,004074431,32416603; 45738588
2,Lawrence Wholesale LLC,Lawernce Wholesale LLC 4353 Excha Lawrence Wholesale LLC,108728254,27867585; 45710781
2,M V And Sons Texas Food,M V And Sons Texas Food Tian Tian Food Service,784228285,45146930; 45746010
2,Mariner Seafood LLC,Mar Lees Seafood LLC Mariner Seafood LLC,037235640,27869391; 33502086



Preview shipper groups


distinct_names,canonical_name,name_list,shipper_panjiva_ids
1,Xiamen Huison Foods Company,Xiamen Huison Foods Company,45342071
1,Guangxi Nanning Baiyang Food Company,Guangxi Nanning Baiyang Food Company,5096630
1,Xinxing Aquatic Products Processing,Xinxing Aquatic Products Processing,29060263
1,Dahuachem International Economic,Dahuachem International Economic,44214093
1,Dalian Oceanstone Zf Group Foods,Dalian Oceanstone Zf Group Foods,44704138
1,Zhangzhou Fuhai Food,Zhangzhou Fuhai Food,33649517
1,Shantou City Qiaofeng Group Company Limited,Shantou City Qiaofeng Group Company Limited,1769168
1,Rongcheng Huiying Foods Company Limited,Rongcheng Huiying Foods Company Limited,30433241
1,Huazhou Xinhai Aquatic Products,Huazhou Xinhai Aquatic Products,44439162
1,Qingdao Ocean Garden Imp And Exp,Qingdao Ocean Garden Imp And Exp,28855377


####

#### 6. (Cleaning) Clusters rows based on clustering logic
#### Clusters consignee / shipper names based on how similar they are (Levenshtein-like, 0.9). Canonical names within clusters are the most frequent names.

In [9]:
import re
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter
from difflib import SequenceMatcher
import pandas as pd

#  Paths 
base = Path("/Users/brant/LLM_Research/DataCleanse/input/us_imports_2015")
src  = base / "condense" / "us_import_2015_combined_std_combined.csv"  # <-- your input
out_dir = base / "cluster"
out_dir.mkdir(parents=True, exist_ok=True)

final_out     = out_dir / "final_combined_std_clustered.csv"
final_output  = out_dir / "final_output.csv"                # <-- renamed from corrections_names.csv
rep_cons_out  = out_dir / "consignee_clusters_report.csv"
rep_ship_out  = out_dir / "shipper_clusters_report.csv"

assert src.exists(), f"Input not found: {src}"

df = pd.read_csv(src, dtype=str, low_memory=False)
for c in ["ConsigneeName","ShipperName"]:
    if c in df.columns:
        df[c] = df[c].fillna("").astype(str).str.strip()
    else:
        df[c] = ""

# Pre-clean: drop a single leading "1" token (e.g., '1. Name' -> 'Name') 
LEADING_ONE_RE = re.compile(r'^\s*1(?=\s|[.\-_/])[\s.\-_/]*', flags=re.IGNORECASE)
def strip_leading_one(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    return LEADING_ONE_RE.sub("", s).strip()

df["ConsigneeName"] = df["ConsigneeName"].apply(strip_leading_one)
df["ShipperName"]   = df["ShipperName"].apply(strip_leading_one)

#  Value / Quantity columns (pick first present) 
VALUE_CANDS = ["ValueOfGoodsUSD","ValueOfGoodsFOBUSD","ExportValue","InvoiceValueUSD",
               "ValueUSD","Value_USD","FOBUSD","CIFUSD","Value"]
QTY_CANDS   = ["GrossWeightKg","NetWeightKg","WeightKg","Weightkg","Quantity (kg)","Quantity_kg","QuantityKg"]

def pick_first_existing(cols, cands):
    for k in cands:
        if k in cols:
            return k
    return None

VAL_COL = pick_first_existing(df.columns, VALUE_CANDS)
QTY_COL = pick_first_existing(df.columns, QTY_CANDS)

def as_float(x):
    """Safe float (strip commas/$, missing -> 0.0)."""
    try:
        return float(str(x).replace(",","").replace("$",""))
    except Exception:
        return 0.0

val_series = df[VAL_COL].map(as_float) if VAL_COL else pd.Series([0.0]*len(df))
qty_series = df[QTY_COL].map(as_float) if QTY_COL else pd.Series([0.0]*len(df))

#  Normalization helper for similarity
PUNCT_TO_SPACE = re.compile(r"[\,\.;:\|\(\)\[\]\{\}/\\\+\=\*\!\?\#\^\"“”‘’`~_–—\-]+")
NUM_TOKEN      = re.compile(r"\b\d+\b")

# Remove generic legal suffixes ONLY FOR MATCHING (not for writing)
# 1) composites like "S. de R.L. de C.V.", "S.A. de C.V." removed as a block
COMPOSITES_TO_STRIP = [
    re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\s*de\s*c\.?\s*v\.?\b", re.I),
    re.compile(r"\bs\.?\s*a\.?\s*de\s*c\.?\s*v\.?\b", re.I),
    re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\b", re.I),
]

# 2) single tokens to drop from matching
LEGAL_STOPWORDS = {
    "inc","incorporated","corp","corporation","co","company",
    "ltd","limited","ltda","pty","plc","llc","llp","gmbh","ag","bv","nv",
    "sa","srl","spa","pte","ptyltd","group","intl","international","int",
    "holdings","holding"
}

# whole-word removal for ALL stopwords (as before)
STOPWORD_WORD_RE = re.compile(
    r"\b(?:" + "|".join(map(re.escape, sorted(LEGAL_STOPWORDS, key=len, reverse=True))) + r")\b",
    re.IGNORECASE
)

# NEW: suffix removal for "large" stopwords only (safer)
LARGE_STOPWORDS = {w for w in LEGAL_STOPWORDS if len(w) >= 5}  # e.g., company, limited, holdings, group, ...
STOPWORD_SUFFIX_RE = re.compile(
    r"(?:" + "|".join(map(re.escape, sorted(LARGE_STOPWORDS, key=len, reverse=True))) + r")$",
    re.IGNORECASE
)

SPACE_RE = re.compile(r"\s+")
PUNCT2SPACE_RE = re.compile(r"[^\w]+", re.UNICODE)

def _strip_large_suffix(token: str) -> str:
    # keep stripping in case of chained suffixes like "acmeholdingsgroup"
    while True:
        new = STOPWORD_SUFFIX_RE.sub("", token)
        if new == token:
            return token
        token = new

# reusable whitespace/punct normalizers (use your existing ones if already defined)
SPACE_RE = re.compile(r"\s+")
PUNCT2SPACE_RE = re.compile(r"[^\w]+", re.UNICODE)

def norm_for_match(s: str, strip_numbers: bool = False) -> str:
    # 1) fold/normalize (keep your existing folding function)
    s = _nfkd_lower(s)

    # 2) map punctuation & separators to spaces so word boundaries are reliable
    s = PUNCT2SPACE_RE.sub(" ", s)

    # 3) remove WHOLE-WORD stopwords in one pass (background only)
    s = STOPWORD_WORD_RE.sub(" ", s)

    toks = (_strip_large_suffix(t) for t in s.split())
    s = " ".join(t for t in toks if t)  # drop empties

    # 4) collapse spaces
    s = SPACE_RE.sub(" ", s).strip()

    if strip_numbers:
        s = NUM_TOKEN.sub("", s)
        s = re.sub(r"\s+"," ", s).strip()

    return s

def length_bucket(s: str, size:int=3) -> int:
    """Rough length bucket to reduce comparisons."""
    return max(0, len(s)//size)

# Clustering with numeric-aware comparison 
CUTOFF = 0.90

def cluster_names_by_similarity(names, counts, cutoff=CUTOFF):
    """Cluster names by similarity, comparing both normal and number-stripped forms."""
    recs = []
    for nm in names:
        n1 = norm_for_match(nm, strip_numbers=False)
        n2 = norm_for_match(nm, strip_numbers=True)
        key = (n1[:1], length_bucket(n1))
        recs.append((nm, n1, n2, key))

    # Block by (first char, length bucket)
    blocks = defaultdict(list)
    for nm, n1, n2, key in recs:
        blocks[key].append((nm, n1, n2))

    clusters = []
    for key, items in blocks.items():
        if len(items) == 1:
            clusters.append([items[0][0]])
            continue

        # Union-Find within block
        parent = list(range(len(items)))
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        def union(a,b):
            ra, rb = find(a), find(b)
            if ra == rb: return
            parent[rb] = ra

        # Pairwise compare within block
        for i in range(len(items)):
            for j in range(i+1, len(items)):
                a, a1, a2 = items[i]
                b, b1, b2 = items[j]
                if abs(len(a1)-len(b1)) > 6:
                    continue
                r_norm = SequenceMatcher(None, a1, b1).ratio()
                r_num  = SequenceMatcher(None, a2, b2).ratio()
                if max(r_norm, r_num) >= cutoff:
                    union(i, j)

        # Collect connected components
        comp = defaultdict(list)
        for idx in range(len(items)):
            comp[find(idx)].append(items[idx][0])
        clusters.extend(comp.values())

    return clusters

def canonical_name(cluster, counts):
    """Pick most frequent as canonical; tie-break lexicographically."""
    vc = Counter({nm: counts.get(nm, 0) for nm in cluster})
    top = max(vc.values())
    return sorted([k for k,v in vc.items() if v == top])[0]

fmt1 = lambda x: f"{x:,.1f}"  # "108,000.0"

def build_for(name_col: str, dataset_label="US Imports 2015"):
    """Cluster one column and build report + corrections + after-series."""
    counts = df[name_col].value_counts()
    unique = list(counts.index)
    clusters = cluster_names_by_similarity(unique, counts.to_dict(), cutoff=CUTOFF)

    name2canon = {}
    rep_rows = []
    for cl in clusters:
        canon = canonical_name(cl, counts.to_dict())
        for nm in cl:
            name2canon[nm] = canon
        rep_rows.append({
            "distinct_names": len(sorted(set(cl))),
            "name_list": "; ".join(sorted(set(cl))),  # CSV-friendly
            "canonical_name": canon,
        })
    report_df = pd.DataFrame(rep_rows).sort_values(
        ["distinct_names","canonical_name"], ascending=[False, True]
    ).reset_index(drop=True)

    agg = pd.DataFrame({
        "before": df[name_col],
        "_val": val_series,
        "_qty": qty_series
    }).groupby("before", dropna=False).agg(
        frequency=("before","size"),
        value_usd=("_val","sum"),
        quantity_kg=("_qty","sum")
    ).reset_index()
    agg["after"] = agg["before"].map(name2canon).fillna(agg["before"])
    agg["Value_USD (of before)"] = agg["value_usd"].map(fmt1)
    agg["Quantity (kg)"]         = agg["quantity_kg"].map(fmt1)

    col_label = "Consignee" if name_col == "ConsigneeName" else "Shipper"
    corr = agg[["before","after","frequency","Value_USD (of before)","Quantity (kg)"]].copy()
    corr.insert(0,"Column Name", col_label)
    corr.insert(0,"Dataset", dataset_label)
    corr["ManualCorrection"] = ""
    corr["Reason (-1,0,1)"]  = ""
    corr["Canonical"]        = ""
    corr = corr[["Dataset","Column Name","before","after","frequency",
                 "Value_USD (of before)","Quantity (kg)",
                 "ManualCorrection","Reason (-1,0,1)","Canonical"]]

    after_series = df[name_col].map(name2canon).fillna(df[name_col])
    return corr, report_df, after_series

# Build for both columns 
corr_cons, rep_cons, after_cons = build_for("ConsigneeName")
corr_ship, rep_ship, after_ship = build_for("ShipperName")

# Combine corrections into ONE output (renamed to final_output.csv)
final_output_df = pd.concat([corr_cons, corr_ship], ignore_index=True)

# Apply canonical names and remove exact dup rows
df_out = df.copy()
df_out["ConsigneeName"] = after_cons
df_out["ShipperName"]   = after_ship
before_rows = len(df_out)
df_out = df_out.drop_duplicates().reset_index(drop=True)
removed = before_rows - len(df_out)

# Save to disk
final_output_df.to_csv(final_output, index=False)
rep_cons.to_csv(rep_cons_out, index=False)
rep_ship.to_csv(rep_ship_out, index=False)
df_out.to_csv(final_out, index=False)

print(f"Files written to: {out_dir}")
print(f"Cutoff used: {CUTOFF:.2f}")

# ---- PREVIEWS (show every name on its own line) ----
def _display_wrapped(df_in: pd.DataFrame, cols, title: str, n: int = 10):
    print(f"\n{title}")
    subset = df_in.loc[:, [c for c in cols if c in df_in.columns]].head(n)
    with pd.option_context('display.max_colwidth', None):
        subset = subset.copy()
        if "name_list" in subset.columns:
            subset["name_list"] = subset["name_list"].astype(str).str.replace("; ", "\n")
        styler = subset.style.set_properties(**{"white-space": "pre-wrap"})
        if hasattr(styler, "hide"):
            styler = styler.hide(axis="index")
        elif hasattr(styler, "hide_index"):
            styler = styler.hide_index()
        display(styler)

print("\nPreview — Consignee clusters report:")
rep_cons_sorted = rep_cons.sort_values(["distinct_names","canonical_name"], ascending=[False, True])
_display_wrapped(rep_cons_sorted, ["distinct_names","canonical_name","name_list"], "Consignee clusters", 10)

print("\nPreview — Shipper clusters report:")
_display_wrapped(rep_ship, ["distinct_names","canonical_name","name_list"], "Shipper clusters", 10)

print("\nPreview — Final output:")
_display_wrapped(final_output_df, ["Dataset","Column Name","before","after","frequency","Value_USD (of before)","Quantity (kg)"], "Final output", 12)


Files written to: \Users\brant\LLM_Research\DataCleanse\input\us_imports_2015\cluster
Cutoff used: 0.90

Preview — Consignee clusters report:

Consignee clusters


distinct_names,canonical_name,name_list
2,Acme Smoked Fish,Acme Smoked Fish Acme Smoked Fish Corporation
2,Anh Quan International Tradig Company,Anh Quan International Tradig Company Anh Quan International Trading
2,C K Global Company Limited,C K Global Company Limited Ck Global Incorporated Company Limited
2,Channel Seafoods International,Channel Sea Food Channel Seafoods International
2,Devi Sea Foods Incorporated,Devi Sea Foods Incorporated Devi Sea Foods Limited
2,Eastern Fish Company LLC,Eastern Fish Company Eastern Fish Company LLC
2,F W Bryce,F W Bryce F W Bryce Incorporated
2,Flegenhaimer,Flegenhaimer Flegenheimer International Incorporated
2,G And L Seafood,G And L Seafood G And L Seafood Incorporated
2,Ha Thanh General Trading Company Limited,Ha Thanh General Trading Company Ha Thanh General Trading Company Limited



Preview — Shipper clusters report:

Shipper clusters


distinct_names,canonical_name,name_list
2,Asian Alliance International Company Limited,Asian Alliance International Asian Alliance International Company Limited
2,Central Seafood,Central Seafood Central Seafoods Incorporated
2,Chengda Development Company Limited,Chenda Development Company Limited Chengda Development Company Limited
2,Dianbai Shengxing Foods Company Limited,Dianbai Shengxing Foods Company Dianbai Shengxing Foods Company Limited
2,Expalsa Exportadora de Alimentos,Expalsa Exportadora de Alimentos Expalsa Exportadora de Alimentos SA
2,Exportadora de Productos Del Oceano,Exportadora de Productos Del Ocea Exportadora de Productos Del Oceano
2,Fujian Huahai Aquatic Company Limited,Fujian Huahai Aquatic Company Limited Fujian Yuehai Aquatic
2,Godaco Seafood Jsc,Godaco Seafood J S C Godaco Seafood Jsc
2,Hainan Zhongyu Seafood Company,Hainan Zhongyu Seafood Company Hainan Zhongyu Seafood Company Limited
2,Hee Chang Trading Company Limited,Hee Chang Rading Company Limited Hee Chang Trading Company Limited



Preview — Final output:

Final output


Dataset,Column Name,before,after,frequency,Value_USD (of before),Quantity (kg)
US Imports 2015,Consignee,,,235,"23,746,600.0","4,464,555.0"
US Imports 2015,Consignee,120008 Tesoros Trading Company,120008 Tesoros Trading Company,1,0.0,"15,649.0"
US Imports 2015,Consignee,21 Century Trading Incorporated,21 Century Trading Incorporated,1,"58,800.0","20,026.0"
US Imports 2015,Consignee,7 Seas Harvest Incorporated,7 Seas Harvest Incorporated,6,"568,000.0","125,734.0"
US Imports 2015,Consignee,7509251 Canada Incorporated,7509251 Canada Incorporated,2,0.0,"47,088.0"
US Imports 2015,Consignee,9782827880 Tel Ex 1 978879 56,9782827880 Tel Ex 1 978879 56,1,"203,000.0","17,978.1"
US Imports 2015,Consignee,99 Cents Only Stores,99 Cents Only Stores,1,"79,900.0","16,065.0"
US Imports 2015,Consignee,A And S Food Trading Incorporated,A And S Food Trading Incorporated,2,"57,700.0","45,669.0"
US Imports 2015,Consignee,A And V Seafood Investments,A And V Seafood Investments,9,"1,521,300.0","199,719.0"
US Imports 2015,Consignee,A Bosa And Company Limited,A Bosa And Company Limited,1,"120,000.0","26,552.0"
